<a href="https://colab.research.google.com/github/danielyoon516/ml-crash-plan/blob/main/02_autograd_and_nn_module.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
import torch

torch.manual_seed(0)
X = torch.randn(100, 3)
w = torch.randn(3,1)
b = torch.randn(1,1)
noise = 0.1 * torch.randn(100,1)
y = X @ w + b + noise
assert y.shape == (100,1)
w_hat = torch.randn(3,1, requires_grad = True)
b_hat = torch.randn(1,1, requires_grad = True)
y_hat = X @ w_hat + b_hat
mse = ((y_hat - y)**2).mean()
assert y_hat.shape == (100,1)
assert (y_hat - y).shape == (100,1)
mse.backward()

N = X.shape[0]
grad_y_hat = (2/N)* (y_hat - y)
assert grad_y_hat.shape == (100,1)
grad_w = X.T @ grad_y_hat
assert grad_w.shape == (3,1)
grad_b = grad_y_hat.sum(dim=0, keepdim=True)
assert grad_b.shape == (1,1)
assert torch.allclose(w_hat.grad, grad_w)
assert torch.allclose(b_hat.grad, grad_b)
print(w_hat.grad)
print(grad_w)




tensor([[-1.3691],
        [ 2.3147],
        [ 2.6323]])
tensor([[-1.3691],
        [ 2.3147],
        [ 2.6323]], grad_fn=<MmBackward0>)


In [4]:
w_hat = torch.randn(3, 1, requires_grad=True)
b_hat = torch.randn(1, 1, requires_grad=True)
lr = 0.1

for step in range(200):
    y_hat = X@w_hat + b_hat
    mse = ((y_hat - y)**2).mean()
    mse.backward()

    with torch.no_grad():
        w_hat -= lr*w_hat.grad
        b_hat -= lr*b_hat.grad
        w_hat.grad.zero_()
        b_hat.grad.zero_()

    if step % 20 == 0:
        print(step, mse.item())

print(mse.item())
print(w_hat, w)
print(b_hat, b)


0 9.939117431640625
20 0.011693534441292286
40 0.009331059642136097
60 0.009329766035079956
80 0.009329764172434807
100 0.009329761378467083
120 0.009329761378467083
140 0.009329761378467083
160 0.009329761378467083
180 0.009329761378467083
0.009329761378467083
tensor([[ 1.2313],
        [-0.7841],
        [-1.0022]], requires_grad=True) tensor([[ 1.2369],
        [-0.7699],
        [-0.9918]])
tensor([[1.7015]], requires_grad=True) tensor([[1.6982]])


# Part 2: nn.Module and optimizer

In [1]:
import torch.nn as nn

class LinearRegression(nn.Module):
    def __init__(self):
        super().__init__()
        self.linear = nn.Linear(3, 1)

    def forward(self, x):
        return self.linear(x)

In [5]:
# Step 2: inspect
model = LinearRegression()
for name, param in model.named_parameters():
    print(name)
    print(param.shape)
# Step 3: set up
loss_fn = nn.MSELoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.1)

# Step 4: train
for step in range(200):
    y_hat = model(X)
    assert y_hat.shape == (100, 1)
    loss = loss_fn(y_hat, y)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()


    if step % 20 == 0:
        print(step, loss.item())

print(loss.item())

# Step 5: compare
print(model.linear.weight.T, w)
print(model.linear.bias, b)


linear.weight
torch.Size([1, 3])
linear.bias
torch.Size([1])
0 5.865856170654297
20 0.01180238090455532
40 0.009333275258541107
60 0.00932976882904768
80 0.009329762309789658
100 0.009329761378467083
120 0.009329761378467083
140 0.009329761378467083
160 0.009329761378467083
180 0.009329761378467083
0.009329761378467083
tensor([[ 1.2313],
        [-0.7841],
        [-1.0022]], grad_fn=<PermuteBackward0>) tensor([[ 1.2369],
        [-0.7699],
        [-0.9918]])
Parameter containing:
tensor([1.7015], requires_grad=True) tensor([[1.6982]])
